Conduct an in-depth time-series decomposition to isolate and interpret trend, seasonality, and residual components, followed by model-based forecasting validation across different temporal segments.

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

In [3]:
dates = pd.date_range(start='2023-01-01', periods=1095, freq='D')

In [4]:
trend_component = np.linspace(100, 300, 1095)
seasonal_component = 30 * np.sin(2 * np.pi * dates.dayofyear / 365.25)
noise_component = np.random.normal(0, 5, 1095)

In [5]:
sales_data = trend_component + seasonal_component + noise_component

In [6]:
df = pd.DataFrame({'date': dates, 'sales': sales_data}).set_index('date')

In [7]:
print(df.head())

                 sales
date                  
2023-01-01  102.999618
2023-01-02  100.523436
2023-01-03  105.151604
2023-01-04  110.226258
2023-01-05  102.137678


In [8]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [9]:
decomp = seasonal_decompose(df['sales'], model='additive', period=365)

In [10]:
df_decomp = df.copy()
df_decomp['trend'] = decomp.trend
df_decomp['seasonal'] = decomp.seasonal
df_decomp['residuals'] = decomp.resid

In [11]:
print("Decomposed temporal components:")
print(df_decomp.dropna().head())

Decomposed temporal components:
                 sales       trend  seasonal  residuals
date                                                   
2023-07-02  127.724406  133.321994 -1.950921  -3.646667
2023-07-03  135.158038  133.492509  1.440344   0.225185
2023-07-04  131.295380  133.680288 -3.271659   0.886750
2023-07-05  135.650076  133.854404  0.285348   1.510325
2023-07-06  134.114157  134.017694 -1.414348   1.510811


In [12]:
train_cutoff = '2025-01-01'

In [13]:
train_df = df_decomp.loc[df_decomp.index < train_cutoff]
test_df = df_decomp.loc[df_decomp.index >= train_cutoff]

In [14]:
print("Training segment shape:", train_df.shape)
print("Testing segment shape:", test_df.shape)

Training segment shape: (731, 4)
Testing segment shape: (364, 4)


In [15]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [16]:
hw_model = ExponentialSmoothing(
    train_df['sales'], 
    trend='add', 
    seasonal='add', 
    seasonal_periods=365
).fit()

c:\python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\python312\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


In [17]:
forecast_sales = hw_model.forecast(len(test_df))

In [18]:
mae = mean_absolute_error(test_df['sales'], forecast_sales)
rmse = np.sqrt(mean_squared_error(test_df['sales'], forecast_sales))

In [19]:
print("--- Forecasting Validation Results ---")
print("Mean Absolute Error (MAE):", round(mae, 3))
print("Root Mean Squared Error (RMSE):", round(rmse, 3))

--- Forecasting Validation Results ---
Mean Absolute Error (MAE): 4.652
Root Mean Squared Error (RMSE): 5.866
